# Urban-CAMELS: build a model-ready dataset

This notebook turns the Urban-CAMELS data (995 urban basins, water years 1990 to 2025) into a dataset ready for a
hydrological model. You decide **what you need**; the notebook then **checks**, **assembles**, **splits** and **packs** it.

| Step | You choose |
|---|---|
| 1 | basins: a preset, explicit IDs, a streamflow completeness rule, area / imperviousness / megaregion filters |
| 2 | time step (daily, hourly or sub-hourly) and period |
| 3 | inputs: forcing products and variables, potential evapotranspiration, annual attributes |
| 4 | target: observed streamflow and its unit |
| 5 | static attributes |
| 6 | gap filling and quality rules |
| 7 | train / validation / test split |
| 8 | output format and folder |

Only the cells marked **EDIT** need changing. Run the notebook from top to bottom.

**Requirements:** Python 3.9 or later with `pandas`, `numpy` and `xarray` (with `netCDF4` or `scipy`); `matplotlib`
for the optional plot. Basins are read in parallel, so 4 to 8 CPU cores help.

**Data:** download the Urban-CAMELS archives and unzip them all into one folder. That folder holds `timeseries/` and
`attributes/`; set `DATA_ROOT` below to it.

In [ ]:
import os, sys, warnings
from pathlib import Path
import pandas as pd, numpy as np
import ucamels_prep as up                     # this notebook sits next to ucamels_prep.py
pd.set_option('display.width', 160); pd.set_option('display.max_columns', 30)

DATA_ROOT = os.environ.get('UCAMELS_DATA') or '/path/to/Urban-CAMELS'     # EDIT: the folder holding timeseries/ and attributes/
print('data:', up.data_root({'data_root': DATA_ROOT}))

## What is available
Products, variables and units at each time step, the dynamic (annual) attributes, and the static attribute groups.

In [ ]:
up.describe_products()

In [ ]:
catalog = up.load_catalog()
print(len(catalog), 'basins')
catalog.head()

Streamflow completeness (percent of the window with data) is tabulated per basin for 13 windows at each resolution.
The table below counts basins meeting a threshold; use it to choose the completeness rule in Step 1.

In [ ]:
res, thr = 'hourly', 90          # try 'daily', 'hourly' or 'subhourly' and any threshold
cov = up.coverage(res)
pd.DataFrame({'basins >= %d %%' % thr: (cov >= thr).sum()}).T

## Step 1 · Basins  — **EDIT**
- `preset`: `'all'` (995) or `'benchmark_247'` (complete daily record WY1990–2025)
- `ids` or `id_file`: your own list of STAIDs (kept only if they are Urban-CAMELS basins)
- `completeness`: `{'resolution': 'daily'|'hourly'|'subhourly', 'window': 'WY2004-2025', 'min_pct': 90}`
- `area_km2`, `impervious_pct`: `(min, max)`; `megaregions`: list; `exclude`: list of STAIDs
- `exclude_flags`: `'negative_hourly_q'`
  (34 tidal or lake-affected gauges with reverse flow in the hourly record; negative values become NaN in the target)

In [ ]:
BASINS = dict(
    preset='benchmark_247',
    # ids=['01094500', '01097000'],
    # completeness=dict(resolution='hourly', window='WY2004-2025', min_pct=90),
    # area_km2=(0, 2000),
    # impervious_pct=(10, 100),
    # exclude_flags=['negative_hourly_q'],
)
basins = up.select_basins(BASINS)
catalog.set_index('STAID').loc[basins].groupby('megaregion').size().sort_values(ascending=False).to_frame('basins').T

## Step 2 · Time step and period  — **EDIT**
- `daily`: one row per date. NWIS daily streamflow is the local calendar day, and hourly products are summed /
  averaged to the basin's local standard-time day. Daymet and gridMET are used as distributed; their daily
  precipitation corresponds best to the 24 h starting near 12 UTC of the date (see the README).
- `hourly`: UTC; each row describes the hour that starts at its label. AORC and NLDAS-2 stamp the end of the hour, so
  the toolkit moves them back one hour; streamflow and MRMS are already hour-beginning.
- `subhourly`: UTC, step `SUBHOURLY_STEP` (`'5min'`, `'10min'`, `'15min'`, `'30min'`). Streamflow is the mean of the
  instantaneous readings inside each step; hourly forcing is held across the steps of its hour.
- The data window is 1989-10-01 to 2025-09-30. Instantaneous streamflow and AORC-era hourly records are dense from
  water year 2008; MRMS radar starts in 2002 and is missing from January 2012 to October 2014.

In [ ]:
RESOLUTION = 'daily'               # 'daily' | 'hourly' | 'subhourly'
SUBHOURLY_STEP = '15min'           # used only when RESOLUTION == 'subhourly'
PERIOD = ('1989-10-01', '2025-09-30')

## Step 3 · Inputs  — **EDIT**
`FORCING` maps a product to the variables you want. Daily step: `daymet`, `gridmet`, and `aorc` / `nldas_hourly`
(aggregated to each basin's local standard-time day). `nldas_daily` is also offered but is summed over UTC days, so it
is shifted by 5 to 8 hours against the local-day streamflow; prefer `nldas_hourly` for a daily model. Hourly and sub-hourly steps: `aorc`, `nldas_hourly`, `mrms`.
`PET` maps a product to methods (`pet_pt` Priestley–Taylor, `pet_pm` Penman–Monteith).
`DYNAMIC` lists annual attributes given at every step (held within each calendar year); housing density is a census
attribute (1990, 2000, 2010, 2020), held (`'step'`) or interpolated (`'linear'`) between census years.

In [ ]:
FORCING = {'daymet': ['prcp', 'tmax', 'tmin', 'srad', 'vp', 'dayl']}
PET = {'daymet': ['pet_pt']}
DYNAMIC = ['imperv', 'class1_urban', 'pop_density', 'housing_density']
HOUSING_INTERP = 'step'            # 'step' | 'linear'

# hourly example:     FORCING = {'aorc': ['prcp', 'tair', 'srad', 'lrad', 'vp', 'wind', 'pres']}; PET = {'aorc': ['pet_pm']}
# sub-hourly example: FORCING = {'mrms': ['prcp'], 'aorc': ['tair', 'srad', 'vp', 'wind']};     PET = {'aorc': ['pet_pm']}

## Step 4 · Target  — **EDIT**
Observed USGS streamflow. `'mm'` gives depth per time step (mm/day, mm/h or mm per sub-hourly step); `'mm/h'` a rate;
`'cms'` or `'cfs'` discharge. Missing observations stay missing (NaN); they are never filled.

In [ ]:
TARGET = dict(unit='mm')

## Step 5 · Static attributes  — **EDIT**
Groups: `location_topography`, `soil`, `geology`, `geology_categorical`, `vegetation`, `climate`,
`climate_categorical`, `hydrologic_signatures`, `dams`, `levees` (or list single `columns`).
- `from_training_period=True` recomputes the climate attributes (and signatures, if chosen) over the training water years
  only, so no test-period information enters the inputs.
- `categorical`: `'codes'` (stable integers over all 995 basins), `'onehot'` or `'drop'`. `fill`: `'median'` or `'none'`.
- Streamflow signatures describe the observed flow; never use them for basins held out in a spatial split.

In [ ]:
STATIC = dict(groups=['location_topography', 'soil', 'geology', 'vegetation', 'climate'],
              columns=[], from_training_period=True, categorical='codes', fill='median')

## Step 6 · Gaps and quality rules  — **EDIT**
Only **inputs** are gap-filled, and only gaps of at most `max_steps` consecutive steps: precipitation with `precip`
(`'zero'`, `'linear'` or `'none'`), every other input (including PET) linearly. AORC also has a few multi-day gaps in
some regions (for example 7 to 11 July 2007); those exceed `max_steps` and drop the basin unless
`drop_if_inputs_missing=False`. Examples: Daymet leaves
31 December of leap years empty (1 step); AORC misses all of 18 June 2024 (24 hourly steps).
A basin is dropped if its target is missing for more than `max_target_missing_pct` percent of the period, or, with
`drop_if_inputs_missing`, if an input still has gaps after filling.

In [ ]:
GAP_FILL = dict(max_steps=2, precip='zero')            # hourly: max_steps=24 ; sub-hourly 15 min: max_steps=96
QC = dict(max_target_missing_pct=100, drop_if_inputs_missing=True)

## Step 7 · Split  — **EDIT**
- `temporal`: every basin in every set; separate train / validation / test periods.
- `spatial`: basins divided into sets (`test_fraction`, `validation_fraction`, `seed`, optional `stratify` by
  `'megaregion'`, `'impervious_tercile'` or `'area_tercile'`); every set covers the whole period.
- `spatial_kfold`: `k` folds of basins, each used once as the test set (`seed`, `stratify`).
- `temporal_spatial`: train basins in the train period, validation basins in the validation period, test basins in the
  test period (prediction in ungauged basins and in time).
Normalisation statistics are always computed on the training basins and training period only.

In [ ]:
SPLIT = dict(method='temporal',
             train=('1989-10-01', '2005-09-30'), validation=('2005-10-01', '2010-09-30'), test=('2010-10-01', '2025-09-30'),
             test_fraction=0.2, validation_fraction=0.1, k=5, seed=42, stratify='megaregion')

## Step 8 · Output  — **EDIT**
- `neuralhydrology`: `time_series/<STAID>.nc`, `attributes/attributes.csv`, basin files per split and a starting
  `nh_config_template.yml` for the NeuralHydrology library.
- `arrays`: `x_dynamic.npy` (basins × steps × inputs), `y_target.npy`, `c_static.npy` and `arrays_index.json` (written
  to disk without holding everything in memory).
- `csv`: one table per basin plus the static table.
Every format also writes the splits, `normalization_stats.json`, `quality_report.csv` and `manifest.json` (every choice
made here, with units and time conventions). Choose a folder with enough free space.

In [ ]:
OUTPUT = dict(dir='urban_camels_dataset', format='neuralhydrology')

## Check the configuration
Nothing is read yet. This catches impossible combinations and warns about risky ones (for example MRMS across its
2012–2014 gap, or streamflow signatures in a spatial split).

In [ ]:
CONFIG = dict(data_root=DATA_ROOT, resolution=RESOLUTION, subhourly_step=SUBHOURLY_STEP, period=PERIOD, basins=BASINS,
              forcing=FORCING, pet=PET, dynamic_attributes=DYNAMIC, housing_interp=HOUSING_INTERP,
              target=TARGET, static_attributes=STATIC, gap_fill=GAP_FILL, qc=QC, split=SPLIT, output=OUTPUT)
up.validate(CONFIG)
up.summary(CONFIG, basins)

## Build
Reads every basin in parallel, places all series on one time axis, fills short input gaps, applies the quality rules
and assembles the static attributes.

In [ ]:
tables, static, report = up.build(CONFIG, basins)

### Quality report
One row per basin: missing target, input gaps before and after filling, steps filled, kept or dropped with the reason.

In [ ]:
print(report.kept.value_counts().rename({True: 'kept', False: 'dropped'}).to_string())
if (~report.kept).any():
    display(report[~report.kept][['reason']].value_counts().to_frame('basins'))
report.describe().loc[['min', '50%', 'max']]

In [ ]:
sid = next(iter(tables))                     # look at one basin
t = tables[sid]
print(sid, t.shape); display(t.describe().T[['count', 'mean', 'min', 'max']])
try:
    import matplotlib.pyplot as plt
    target = [c for c in t.columns if c.startswith('streamflow_')][0]
    prcp = [c for c in t.columns if c.endswith('__prcp') or c.endswith('__pr')]
    fig, ax = plt.subplots(2, 1, figsize=(11, 4.5), sharex=True)
    if prcp: ax[0].plot(t.index, t[prcp[0]], lw=0.5, color='tab:blue'); ax[0].set_ylabel(prcp[0])
    ax[1].plot(t.index, t[target], lw=0.5, color='tab:red'); ax[1].set_ylabel(target)
    plt.tight_layout(); plt.show()
except ImportError:
    pass

## Split

In [ ]:
split = up.make_split(list(tables), CONFIG)
if 'sets' in split:
    display(pd.DataFrame({k: {'basins': len(v['basins']), 'start': v['period'][0], 'end': v['period'][1]}
                          for k, v in split['sets'].items()}).T)
else:
    display(pd.DataFrame({f: {'train basins': len(v['train']), 'test basins': len(v['test'])}
                          for f, v in split['folds'].items()}).T)

## Write the package

In [ ]:
out = up.write_package(tables, static, report, split, CONFIG)
for p in sorted(out.iterdir()):
    n = len(list(p.iterdir())) if p.is_dir() else None
    print(f'{p.name:32s}' + (f'{n} files' if n is not None else f'{p.stat().st_size / 1e6:.2f} MB'))

## Reading the package back

In [ ]:
import json
m = json.load(open(out / 'manifest.json'))
print('inputs :', m['inputs']); print('target :', m['target']); print('units  :', m['units'])
if CONFIG['output']['format'] == 'neuralhydrology':
    import xarray as xr
    ds = xr.open_dataset(out / 'time_series' / f'{sid}.nc'); print(ds)
elif CONFIG['output']['format'] == 'arrays':
    x = np.load(out / 'x_dynamic.npy', mmap_mode='r'); print('x', x.shape, '| y', np.load(out / 'y_target.npy', mmap_mode='r').shape)

## Notes
- **Time.** Daily rows are dates (streamflow = local calendar day). Hourly and sub-hourly rows are UTC and describe the
  step that starts at the label; AORC and NLDAS-2 values are moved back one hour from their end-of-hour stamps.
- **Streamflow.** USGS observations; provisional values are included as published. Missing values remain NaN — use a
  loss that ignores them.
- **Leakage.** Normalisation statistics come from the training data only; use `from_training_period=True` for climate
  attributes; do not feed streamflow signatures for basins you test as ungauged.
- **National Water Model** simulations are distributed as a benchmark and are deliberately not offered as an input.
- **Provenance.** `manifest.json` records every choice; keep it with any results. Toolkit:
  `ucamels_prep.py`; tests: `tests/test_toolkit.py`.